# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 37 · Opponent traffic and closest approach: full three-arm experiment

Prepared, not executed here. Three matched models: availability control, six-channel core, twelve-channel full. All share the existing motion representation, masks, initialization procedure and exposure. Reused evaluation games are not independent confirmation.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json, sys, os, subprocess
import plotly.io as pio
import plotly.graph_objects as go
KIT = Path('/home/sagemaker-user/nfl_feature_rounds16_17')
ROUND = 17
OUT = Path(f'/home/sagemaker-user/nfl-feature-round{ROUND}-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space; do not create another environment.')
for _name in [p.stem for p in KIT.glob('*.py')]:
    _loaded = sys.modules.get(_name)
    _file = getattr(_loaded, '__file__', None)
    if _file and Path(_file).resolve().parent != KIT.resolve():
        raise RuntimeError('Restart this kernel: a different kit owns '+_name)
previous = globals().get('_NFL_ACTIVE_KIT')
if previous and previous != str(KIT):
    raise RuntimeError('Restart this kernel before changing kits.')
_NFL_ACTIVE_KIT = str(KIT)
os.chdir(KIT)
if str(KIT) not in sys.path: sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
PROCEED = False

def run(stage, *options):
    if stage in ('prepare','runtime','profile','train','evaluate','replay') and not PROCEED:
        print('REVIEW PENDING: no scientific stage executed. Return the readiness report.')
        return False
    command = [str(PY),str(KIT/'run_round.py'),stage,'--round',str(ROUND),*options]
    # The existing launcher enforces its stage cap, locks and worker-group shutdown.
    process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                               text=True,bufsize=1,cwd=KIT)
    try:
        for line in process.stdout: print(line,end='')
        result = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            raise RuntimeError('Launcher did not exit promptly; stop here and inspect its log. Do not restart concurrently.')
        raise
    if result:
        raise RuntimeError(f'{stage} stopped ({result}). Use the report cell; do not rerun unchanged failures.')
    return True

def show(fig, name):
    visuals.save(fig,OUT,name).show()

def collect():
    completed = subprocess.run([str(PY),'/home/sagemaker-user/nfl_workspace.zip','bundle'],cwd=KIT)
    if completed.returncode: raise RuntimeError('Report collection stopped; return the printed diagnostic.')

def history_plot(which, roles=False):
    summary = visuals.read(OUT.parent/f'nfl-feature-round{which}-results'/'summary.json')
    fig = go.Figure()
    if roles:
        for arm in ('mask','core','full','preserved_tree'):
            rows=[x for x in summary['slices'] if x['arm']==arm and x['slice'].startswith('role_')]
            fig.add_bar(name=arm,x=[x['slice'] for x in rows],y=[x['rmse'] for x in rows])
        fig.update_layout(barmode='group')
    else:
        names=list(summary['metrics'])
        fig.add_bar(x=names,y=[summary['metrics'][x]['rmse'] for x in names])
    return visuals.style(fig,f'Completed Round {which} — reused internal games, not Kaggle',
                         'Role code: 0 receiver, 1 coverage' if roles else 'Arm','Coordinate RMSE (yards)')


## Review decision
No active new release is supplied. Pending readiness keeps all scientific cells inactive. Do not edit or invent a release to bypass this gate.

In [ ]:
from readiness import status
ready = status(SimpleNamespace(kit=KIT,out=OUT,round_no=ROUND))
PROCEED = ready['ready']
print(json.dumps(ready,indent=2))
if not PROCEED:
    print('This is a decision gate, not a model error. Finish both research notebooks and return nfl_workspace_report.zip.')


## Preparation and offline runtime checks
Uses the same verified CPU lock and cached packages. No installation, remote call, or new environment version is requested.

In [ ]:
run('prepare')
run('runtime')

## Training-only cost guard
Twenty-four disposable steps across the three arms. Require `family_profile_passed`. A budget stop requires review, not a larger cap.

In [ ]:
run('profile')

## Fixed-exposure arms
Run each training cell separately. A model completion receipt is not a feature-value claim.

In [ ]:
run('train','--arm','mask')

In [ ]:
run('train','--arm','core')

In [ ]:
run('train','--arm','full')

## Official coordinate RMSE and ablations
Use all requested rows. Differences and adjusted intervals apply to six planned comparisons across these two studies only, not to every adaptive decision in prior rounds.

In [ ]:
if PROCEED:
    run('evaluate')
    show(visuals.learning(OUT),'training_objectives')
    show(visuals.metrics(OUT),'matched_metrics')
    show(visuals.contrasts(OUT),'paired_contrasts')
    show(visuals.horizons(OUT),'horizon_errors')
else:
    print('No scoring or model plots: readiness review is pending.')

## Fresh-process replay and export
Replay must report zero new optimizer steps. It cannot fit a missing model. Save and reopen the notebook to verify the four inline figures remain visible.

In [ ]:
if PROCEED: run('replay')
run('report')
collect()